# Installation

In [1]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.2 MB/s et

In [2]:
!pip install sentence-transformers openai

# Initializing Chroma

In [3]:
import chromadb

client = chromadb.Client()

In [4]:
client = chromadb.PersistentClient(path="./chroma_db")

# Creating a Collection

In [5]:
collection = client.get_or_create_collection(name="my_docs")

# Adding Data to Chromadb collection

In [6]:
documents = [
    "COVID-19 vaccines have been shown to reduce severe illness and hospitalization rates.",
    "Diabetes management requires regular monitoring of blood glucose levels and lifestyle adjustments.",
    "Hypertension is a leading risk factor for heart disease and stroke, requiring timely intervention.",
    "MRI scans provide detailed images of soft tissues, helping in the diagnosis of neurological disorders.",
    "Antibiotic resistance is a growing concern in the treatment of bacterial infections.",
    "Telemedicine allows patients to consult doctors remotely, improving access to healthcare services.",
    "Cancer immunotherapy leverages the body's immune system to target and destroy malignant cells.",
    "Mental health disorders, such as depression and anxiety, affect millions and require comprehensive care.",
    "Wearable devices can track heart rate, sleep patterns, and activity levels for personalized health insights.",
    "Genetic testing can help identify individuals at risk for hereditary diseases and guide preventive strategies."
]

# Metadata for filtering/searching
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# Unique IDs for each document
ids = [f"doc{i+1}" for i in range(len(documents))]

# Example: Adding to a Chroma collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 73.5MiB/s]


# Querying the Database

In [7]:
query = "How does cancer immunotherapy work?"

results = collection.query(
    query_texts=[query],
    n_results=2 #top k results
)

print(results)

{'ids': [['doc7', 'doc1']], 'embeddings': None, 'documents': [["Cancer immunotherapy leverages the body's immune system to target and destroy malignant cells.", 'COVID-19 vaccines have been shown to reduce severe illness and hospitalization rates.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'NCI', 'topic': 'oncology'}, {'topic': 'vaccines', 'source': 'WHO'}]], 'distances': [[0.2975139617919922, 1.3732753992080688]]}


# Filtering by Metadata


In [8]:
query = "How can wearable devices track heart health?"

results = collection.query(
    query_texts=[query],
    n_results=2,
    where={"topic": "wearables"}  # filter by metadata
)

print(results["documents"])

[['Wearable devices can track heart rate, sleep patterns, and activity levels for personalized health insights.']]


# Using Custom Embeddings (Option 1)

In [9]:
# Import Chroma and embedding function
import chromadb
from chromadb.utils import embedding_functions

# ✅ Define the embedding model (Sentence Transformer)
# all-MiniLM-L6-v2 is lightweight, fast, and good for semantic medical text search
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ✅ Initialize a persistent Chroma client (data stored on disk)
client = chromadb.PersistentClient(path="./medical_chroma_db")

# ✅ Create (or get) a collection with embedding support
collection = client.get_or_create_collection(
    name="medical_documents",
    embedding_function=sentence_transformer_ef
)

# ✅ Medical domain documents
documents = [
    "COVID-19 vaccines have been shown to reduce severe illness and hospitalization rates.",
    "Diabetes management requires regular monitoring of blood glucose levels and lifestyle adjustments.",
    "Hypertension is a leading risk factor for heart disease and stroke, requiring timely intervention.",
    "MRI scans provide detailed images of soft tissues, helping in the diagnosis of neurological disorders.",
    "Antibiotic resistance is a growing concern in the treatment of bacterial infections.",
    "Telemedicine allows patients to consult doctors remotely, improving access to healthcare services.",
    "Cancer immunotherapy leverages the body's immune system to target and destroy malignant cells.",
    "Mental health disorders, such as depression and anxiety, affect millions and require comprehensive care.",
    "Wearable devices can track heart rate, sleep patterns, and activity levels for personalized health insights.",
    "Genetic testing can help identify individuals at risk for hereditary diseases and guide preventive strategies."
]

# ✅ Corresponding metadata for filtering/searching
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# ✅ Unique IDs for each document
ids = [f"doc{i+1}" for i in range(len(documents))]

# ✅ Add data to the Chroma collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("✅ Medical documents successfully added to Chroma!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Medical documents successfully added to Chroma!


In [10]:
query = "How can wearable devices help with heart health?"

results = collection.query(
    query_texts=[query],
    n_results=2
)

print("🔍 Query Results:")
print(results)

🔍 Query Results:
{'ids': [['doc9', 'doc3']], 'embeddings': None, 'documents': [['Wearable devices can track heart rate, sleep patterns, and activity levels for personalized health insights.', 'Hypertension is a leading risk factor for heart disease and stroke, requiring timely intervention.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'topic': 'wearables', 'source': 'Stanford'}, {'source': 'MayoClinic', 'topic': 'cardiology'}]], 'distances': [[0.3084343671798706, 0.6405394077301025]]}


# Using Custom Embeddings (Option 2)

In [11]:
# Save medical documents to Chroma using OpenAI embeddings manually
import chromadb
import os
from dotenv import load_dotenv
import openai

# ✅ Load environment variables
load_dotenv()
openai_api_key = os.getenv("OPEN_AI_KEY")

# ✅ Set up OpenAI API key
openai.api_key = openai_api_key

# ✅ Custom function to generate embeddings using OpenAI API
def generate_embedding(text: str, model: str = "text-embedding-3-small") -> list:
    """
    Generate a vector embedding for a given text using OpenAI embeddings API.
    """
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    embedding_vector = response.data[0].embedding
    return embedding_vector

# ✅ Initialize a persistent Chroma client
client = chromadb.PersistentClient(path="./medical_chroma_openai_db")

# ✅ Create or get collection (without Chroma embedding function)
collection = client.get_or_create_collection(
    name="medical_documents_custom"
)

# ✅ Medical domain documents
documents = [
    "COVID-19 vaccines have been shown to reduce severe illness and hospitalization rates.",
    "Diabetes management requires regular monitoring of blood glucose levels and lifestyle adjustments.",
    "Hypertension is a leading risk factor for heart disease and stroke, requiring timely intervention.",
    "MRI scans provide detailed images of soft tissues, helping in the diagnosis of neurological disorders.",
    "Antibiotic resistance is a growing concern in the treatment of bacterial infections.",
    "Telemedicine allows patients to consult doctors remotely, improving access to healthcare services.",
    "Cancer immunotherapy leverages the body's immune system to target and destroy malignant cells.",
    "Mental health disorders, such as depression and anxiety, affect millions and require comprehensive care.",
    "Wearable devices can track heart rate, sleep patterns, and activity levels for personalized health insights.",
    "Genetic testing can help identify individuals at risk for hereditary diseases and guide preventive strategies."
]

# ✅ Corresponding metadata
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# ✅ Unique IDs
ids = [f"doc{i+1}" for i in range(len(documents))]

# ✅ Generate embeddings manually
embeddings = [generate_embedding(doc) for doc in documents]

# ✅ Add data to Chroma collection with precomputed embeddings
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)

print("✅ Medical documents successfully added to Chroma (custom embeddings)!")

# ✅ Example query embedding
query = "How can wearable devices help with heart health?"
query_embedding = generate_embedding(query)

# ✅ Perform similarity search using the query embedding
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print("🔍 Query Results:")
print(results)

# Updating and Deleting Data

## Update (Overwrite by ID)

In [12]:
collection.update(
    ids=["doc1"],
    documents=["Chroma is a vector database optimized for AI apps."]
)

## Delete

In [13]:
collection.delete(ids=["doc3"])

# Integration with LLMs (RAG Example)

In [14]:
from openai import OpenAI
client_llm = OpenAI(api_key=openai_api_key)

query = "How can wearable devices help with heart health?"

query_embedding = generate_embedding(query)

results = collection.query(query_embeddings=[query_embedding], n_results=3)
context = "\n".join(results["documents"][0])

prompt = f"""
Answer the following question using the context below.

Context:
{context}

Question: {query}
"""

response = client_llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)